In [10]:
# FAMILY FINANCE DASHBOARD

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

TRANSACTION_FILE = r"S:\FINANCE DASHBOARD\Finance_dashboard_large_dataset.csv"
MONTHLY_FILE = r"S:\FINANCE DASHBOARD\monthly_summary.csv"
CATEGORY_FILE = r"S:\FINANCE DASHBOARD\category_summary.csv"

ld = pd.read_csv(TRANSACTION_FILE)
ms = pd.read_csv(MONTHLY_FILE)
cs = pd.read_csv(CATEGORY_FILE)

ld["Date"] = pd.to_datetime(ld["Date"], errors="coerce")
ld["Amount_INR"] = pd.to_numeric(ld["Amount_INR"], errors="coerce").fillna(0)

for col in ["Transaction_Type", "Category", "Merchant", "Payment_Method",
            "City", "Status", "Month", "Quarter", "Weekday", "Account_ID"]:
    if col in ld.columns:
        ld[col] = ld[col].astype(str).str.strip()

# Derived numerical columns
ld["Is_Income"] = (ld["Transaction_Type"].str.lower() == "income")
ld["Is_Expense"] = (ld["Transaction_Type"].str.lower() == "expense")

ld["Expense_Amount"] = np.where(ld["Is_Expense"], ld["Amount_INR"], 0)
ld["Income_Amount"] = np.where(ld["Is_Income"], ld["Amount_INR"], 0)

# Date-derived columns
ld["Month_Date"] = ld["Date"].dt.to_period("M").dt.to_timestamp()
ld["Month_Label"] = ld["Month_Date"].dt.strftime("%b-%Y")
ld["Day"] = ld["Date"].dt.day
ld["Day_of_Month"] = ld["Date"].dt.day
ld["Week_Number"] = ld["Date"].dt.isocalendar().week.astype(int)

expense_df = ld[ld["Is_Expense"]].copy()
income_df = ld[ld["Is_Income"]].copy()

print("FAMILY FINANCE DASHBOARD\n")
print(f"Transactions loaded : {len(ld):,}")
print(f"Columns             : {len(ld.columns):,}")
print(f"Date range          : {ld['Date'].min().date()} → {ld['Date'].max().date()}")
print(f"Income transactions : {len(income_df):,}")
print(f"Expense transactions: {len(expense_df):,}")
print(f"Categories          : {ld['Category'].nunique():,}")
print(f"Accounts             : {ld['Account_ID'].nunique():,}")
print(f"Merchants            : {ld['Merchant'].nunique():,}")
print(f"Cities               : {ld['City'].nunique():,}")
def inr(x, pos=None):
    return f"₹{x:,.0f}"

def money_axis():
    plt.gca().yaxis.set_major_formatter(FuncFormatter(inr))

def finish(title, xlabel=None, ylabel=None, rotation=0, legend=False):
    plt.title(title, fontsize=16, fontweight="bold", pad=14)
    if xlabel:
        plt.xlabel(xlabel)
    if ylabel:
        plt.ylabel(ylabel)
    if rotation:
        plt.xticks(rotation=rotation, ha="right")
    if legend:
        plt.legend()
    plt.grid(axis="y", alpha=0.20)
    plt.tight_layout()
    plt.show()

def monthly_table():
    x = ld.groupby(["Month_Date", "Transaction_Type"])["Amount_INR"].sum().unstack(fill_value=0)
    if "Income" not in x.columns:
        x["Income"] = 0
    if "Expense" not in x.columns:
        x["Expense"] = 0
    x["Net"] = x["Income"] - x["Expense"]
    x["Savings_Rate"] = np.where(x["Income"] != 0, x["Net"] / x["Income"] * 100, np.nan)
    return x.sort_index()

def expense_category():
    return expense_df.groupby("Category")["Amount_INR"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)

def income_category():
    return income_df.groupby("Category")["Amount_INR"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False)


FAMILY FINANCE DASHBOARD

Transactions loaded : 250,000
Columns             : 23
Date range          : 2021-01-01 → 2026-06-30
Income transactions : 48,870
Expense transactions: 201,130
Categories          : 19
Accounts             : 8
Merchants            : 50
Cities               : 10


In [11]:
def chart_monthly_income_expense():
    x = monthly_table()
    plt.figure(figsize=(15, 7))
    plt.plot(x.index, x["Income"], marker="o", linewidth=2, label="Income")
    plt.plot(x.index, x["Expense"], marker="o", linewidth=2, label="Expense")
    money_axis()
    finish("Monthly Income vs Expense", "Month", "Amount (INR)", rotation=45, legend=True)

def chart_monthly_net():
    x = monthly_table()
    plt.figure(figsize=(15, 6))
    plt.bar(x.index, x["Net"])
    money_axis()
    finish("Monthly Net Cash Flow", "Month", "Net Cash Flow (INR)", rotation=45)

def chart_cumulative_cashflow():
    x = monthly_table()
    x["Cumulative_Net"] = x["Net"].cumsum()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x["Cumulative_Net"], linewidth=2.5)
    money_axis()
    finish("Cumulative Net Cash Flow", "Month", "Cumulative INR", rotation=45)

def chart_savings_rate():
    x = monthly_table()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x["Savings_Rate"], marker="o", linewidth=2)
    plt.axhline(0, linewidth=1)
    finish("Monthly Savings Rate", "Month", "Savings Rate (%)", rotation=45)

def chart_expense_category():
    x = expense_category()["sum"].sort_values()
    plt.figure(figsize=(11, 8))
    plt.barh(x.index, x.values)
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("Total Expense by Category", "Amount (INR)", "Category")

def chart_expense_category_share():
    x = expense_category()["sum"].head(12)
    plt.figure(figsize=(9, 9))
    plt.pie(x.values, labels=x.index, autopct="%1.1f%%", startangle=90)
    plt.title("Top Expense Category Share", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

def chart_income_category():
    x = income_category()["sum"].sort_values()
    plt.figure(figsize=(10, 7))
    plt.barh(x.index, x.values)
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("Income by Category", "Amount (INR)", "Category")

def chart_payment_method():
    x = expense_df.groupby("Payment_Method")["Amount_INR"].sum().sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    plt.bar(x.index, x.values)
    money_axis()
    finish("Expense by Payment Method", "Payment Method", "Amount (INR)", rotation=30)

def chart_city():
    x = expense_df.groupby("City")["Amount_INR"].sum().sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    plt.bar(x.index, x.values)
    money_axis()
    finish("Expense by City", "City", "Amount (INR)", rotation=30)

def chart_account():
    x = expense_df.groupby("Account_ID")["Amount_INR"].sum().sort_values(ascending=False)
    plt.figure(figsize=(11, 6))
    plt.bar(x.index, x.values)
    money_axis()
    finish("Expense by Account", "Account", "Amount (INR)", rotation=30)

def chart_merchant():
    x = expense_df.groupby("Merchant")["Amount_INR"].sum().nlargest(15).sort_values()
    plt.figure(figsize=(11, 8))
    plt.barh(x.index, x.values)
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("Top 15 Merchants by Expense", "Amount (INR)", "Merchant")

def chart_weekday():
    order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    x = expense_df.groupby("Weekday")["Amount_INR"].sum().reindex(order)
    plt.figure(figsize=(10, 6))
    plt.bar(x.index, x.values)
    money_axis()
    finish("Expense by Day of Week", "Day", "Amount (INR)", rotation=25)

def chart_monthly_transaction_count():
    x = ld.groupby("Month_Date").size()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x.values, marker="o")
    finish("Monthly Transaction Volume", "Month", "Number of Transactions", rotation=45)

def chart_monthly_expense_count():
    x = expense_df.groupby("Month_Date").size()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x.values, marker="o")
    finish("Monthly Expense Transaction Count", "Month", "Expense Transactions", rotation=45)

def chart_average_expense():
    x = expense_df.groupby("Month_Date")["Amount_INR"].mean()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x.values, marker="o")
    money_axis()
    finish("Average Expense per Transaction", "Month", "Average Expense (INR)", rotation=45)

def chart_median_expense():
    x = expense_df.groupby("Month_Date")["Amount_INR"].median()
    plt.figure(figsize=(15, 6))
    plt.plot(x.index, x.values, marker="o")
    money_axis()
    finish("Median Expense per Transaction", "Month", "Median Expense (INR)", rotation=45)

def chart_expense_distribution():
    plt.figure(figsize=(11, 6))
    plt.hist(expense_df["Amount_INR"], bins=60)
    plt.xlabel("Transaction Amount (INR)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Expense Transactions", fontsize=16, fontweight="bold")
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

def chart_income_distribution():
    plt.figure(figsize=(11, 6))
    plt.hist(income_df["Amount_INR"], bins=60)
    plt.xlabel("Transaction Amount (INR)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Income Transactions", fontsize=16, fontweight="bold")
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

def chart_quarterly():
    x = ld.groupby(["Year", "Quarter", "Transaction_Type"])["Amount_INR"].sum().unstack(fill_value=0)
    if "Income" not in x.columns:
        x["Income"] = 0
    if "Expense" not in x.columns:
        x["Expense"] = 0
    x["Net"] = x["Income"] - x["Expense"]
    labels = [f"{y}-{q}" for y, q in x.index]

    plt.figure(figsize=(14, 6))
    plt.plot(labels, x["Income"].values, marker="o", label="Income")
    plt.plot(labels, x["Expense"].values, marker="o", label="Expense")
    plt.plot(labels, x["Net"].values, marker="o", label="Net")
    money_axis()
    finish("Quarterly Financial Performance", "Quarter", "Amount (INR)", rotation=45, legend=True)

def chart_yearly():
    x = ld.groupby(["Year", "Transaction_Type"])["Amount_INR"].sum().unstack(fill_value=0)
    if "Income" not in x.columns:
        x["Income"] = 0
    if "Expense" not in x.columns:
        x["Expense"] = 0
    x["Net"] = x["Income"] - x["Expense"]

    plt.figure(figsize=(11, 6))
    width = 0.25
    idx = np.arange(len(x))
    plt.bar(idx - width, x["Income"], width, label="Income")
    plt.bar(idx, x["Expense"], width, label="Expense")
    plt.bar(idx + width, x["Net"], width, label="Net")
    plt.xticks(idx, x.index)
    money_axis()
    finish("Yearly Income, Expense & Net", "Year", "Amount (INR)", legend=True)

def chart_category_month_heatmap():
    pivot = expense_df.pivot_table(
        index="Category", columns="Month_Date",
        values="Amount_INR", aggfunc="sum", fill_value=0
    )
    pivot = pivot.loc[pivot.sum(axis=1).nlargest(12).index]

    plt.figure(figsize=(18, 9))
    plt.imshow(pivot.values, aspect="auto")
    plt.yticks(range(len(pivot.index)), pivot.index)
    step = max(1, len(pivot.columns)//12)
    plt.xticks(range(0, len(pivot.columns), step),
               [d.strftime("%b-%y") for d in pivot.columns[::step]], rotation=45)
    plt.colorbar(label="Expense (INR)")
    plt.title("Category × Month Expense Heatmap", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

def chart_category_frequency():
    x = expense_df["Category"].value_counts().sort_values()
    plt.figure(figsize=(10, 8))
    plt.barh(x.index, x.values)
    finish("Number of Expense Transactions by Category", "Transactions", "Category")

def chart_status():
    x = ld["Status"].value_counts()
    plt.figure(figsize=(9, 6))
    plt.bar(x.index, x.values)
    finish("Transaction Status", "Status", "Transactions", rotation=25)

def chart_daily_expense():
    x = expense_df.groupby("Date")["Amount_INR"].sum()
    plt.figure(figsize=(16, 6))
    plt.plot(x.index, x.values, linewidth=1)
    money_axis()
    finish("Daily Expense Trend", "Date", "Expense (INR)", rotation=45)

def chart_daily_income():
    x = income_df.groupby("Date")["Amount_INR"].sum()
    plt.figure(figsize=(16, 6))
    plt.plot(x.index, x.values, linewidth=1)
    money_axis()
    finish("Daily Income Trend", "Date", "Income (INR)", rotation=45)

def chart_top_transactions():
    x = expense_df.nlargest(20, "Amount_INR")[["Date", "Merchant", "Category", "Amount_INR"]].copy()
    x["Label"] = x["Merchant"] + " | " + x["Category"]
    x = x.sort_values("Amount_INR")
    plt.figure(figsize=(12, 9))
    plt.barh(x["Label"], x["Amount_INR"])
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("20 Largest Expense Transactions", "Amount (INR)", "Merchant | Category")

def chart_category_average():
    x = expense_df.groupby("Category")["Amount_INR"].mean().sort_values()
    plt.figure(figsize=(11, 8))
    plt.barh(x.index, x.values)
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("Average Transaction Amount by Category", "Average INR", "Category")

def chart_account_transaction_count():
    x = ld["Account_ID"].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(11, 6))
    plt.bar(x.index, x.values)
    finish("Transactions by Account", "Account", "Transactions", rotation=30)

def chart_city_transaction_count():
    x = ld["City"].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(11, 6))
    plt.bar(x.index, x.values)
    finish("Transactions by City", "City", "Transactions", rotation=30)

def chart_category_income_expense():
    x = pd.crosstab(
        ld["Category"], ld["Transaction_Type"],
        values=ld["Amount_INR"], aggfunc="sum"
    ).fillna(0).sort_values("Expense", ascending=False)

    if "Income" not in x:
        x["Income"] = 0
    if "Expense" not in x:
        x["Expense"] = 0

    x = x.head(15)
    idx = np.arange(len(x))
    width = 0.38

    plt.figure(figsize=(13, 8))
    plt.barh(idx - width/2, x["Income"], width, label="Income")
    plt.barh(idx + width/2, x["Expense"], width, label="Expense")
    plt.yticks(idx, x.index)
    plt.gca().xaxis.set_major_formatter(FuncFormatter(inr))
    finish("Income vs Expense by Category", "Amount (INR)", "Category", legend=True)

def chart_rolling_expense():
    daily = expense_df.groupby("Date")["Amount_INR"].sum().sort_index()
    rolling = daily.rolling(30, min_periods=1).mean()

    plt.figure(figsize=(16, 6))
    plt.plot(daily.index, daily.values, alpha=0.35, label="Daily Expense")
    plt.plot(rolling.index, rolling.values, linewidth=2.5, label="30-Day Average")
    money_axis()
    finish("Daily Expense with 30-Day Rolling Average", "Date", "INR", rotation=45, legend=True)

def chart_top_categories_month():
    x = expense_category()["sum"].nlargest(10).index
    pivot = expense_df[expense_df["Category"].isin(x)].pivot_table(
        index="Month_Date", columns="Category",
        values="Amount_INR", aggfunc="sum", fill_value=0
    )

    plt.figure(figsize=(16, 8))
    for c in pivot.columns:
        plt.plot(pivot.index, pivot[c], marker="o", linewidth=1.5, label=c)
    money_axis()
    finish("Top Expense Categories Over Time", "Month", "Amount (INR)", rotation=45, legend=True)


In [12]:
def home_page():
    total_income = income_df["Amount_INR"].sum()
    total_expense = expense_df["Amount_INR"].sum()
    net = total_income - total_expense

    avg_expense = expense_df["Amount_INR"].mean()
    median_expense = expense_df["Amount_INR"].median()

    monthly = monthly_table()
    best_month = monthly["Net"].idxmax()
    worst_month = monthly["Net"].idxmin()

    top_category = expense_category().index[0]
    top_category_value = expense_category().iloc[0]["sum"]

    top_merchant = expense_df.groupby("Merchant")["Amount_INR"].sum().idxmax()
    top_merchant_value = expense_df.groupby("Merchant")["Amount_INR"].sum().max()

    top_city = expense_df.groupby("City")["Amount_INR"].sum().idxmax()
    top_city_value = expense_df.groupby("City")["Amount_INR"].sum().max()

    print("FAMILY FINANCE DASHBOARD")

    print("\n📊 OVERALL FINANCIAL POSITION")
    print(f"Total Transactions       : {len(ld):,}")
    print(f"Total Income             : ₹{total_income:,.2f}")
    print(f"Total Expenses           : ₹{total_expense:,.2f}")
    print(f"Net Cash Flow            : ₹{net:,.2f}")
    print(f"Income / Expense Ratio   : {(total_income / total_expense if total_expense else np.nan):.2f}x")
    print(f"Savings Rate             : {(net / total_income * 100 if total_income else np.nan):.2f}%")

    print("\n📅 TIME PERIOD")
    print(f"First Transaction        : {ld['Date'].min().date()}")
    print(f"Last Transaction         : {ld['Date'].max().date()}")
    print(f"Months Covered           : {ld['Month_Date'].nunique():,}")
    print(f"Years Covered            : {ld['Year'].nunique():,}")

    print("\n🏆 SPENDING HIGHLIGHTS")
    print(f"Largest Expense Category : {top_category} — ₹{top_category_value:,.2f}")
    print(f"Top Merchant             : {top_merchant} — ₹{top_merchant_value:,.2f}")
    print(f"Top City                 : {top_city} — ₹{top_city_value:,.2f}")
    print(f"Average Expense          : ₹{avg_expense:,.2f}")
    print(f"Median Expense           : ₹{median_expense:,.2f}")

    print("\n📈 MONTHLY PERFORMANCE")
    print(f"Highest Net Month        : {best_month.strftime('%B %Y')} — ₹{monthly.loc[best_month, 'Net']:,.2f}")
    print(f"Lowest Net Month         : {worst_month.strftime('%B %Y')} — ₹{monthly.loc[worst_month, 'Net']:,.2f}")
    print(f"Best Savings Rate        : {monthly['Savings_Rate'].max():.2f}%")
    print(f"Worst Savings Rate       : {monthly['Savings_Rate'].min():.2f}%")

    print("\n🗂️ DATABASE DIMENSIONS")
    print(f"Categories               : {ld['Category'].nunique():,}")
    print(f"Accounts                 : {ld['Account_ID'].nunique():,}")
    print(f"Merchants                : {ld['Merchant'].nunique():,}")
    print(f"Cities                   : {ld['City'].nunique():,}")
    print(f"Payment Methods          : {ld['Payment_Method'].nunique():,}")
    print(f"Statuses                 : {ld['Status'].nunique():,}")

    print("Use the dashboard menu in Cell 4 to open any analysis.")


home_page()


FAMILY FINANCE DASHBOARD

📊 OVERALL FINANCIAL POSITION
Total Transactions       : 250,000
Total Income             : ₹1,943,645,506.53
Total Expenses           : ₹624,333,954.11
Net Cash Flow            : ₹1,319,311,552.42
Income / Expense Ratio   : 3.11x
Savings Rate             : 67.88%

📅 TIME PERIOD
First Transaction        : 2021-01-01
Last Transaction         : 2026-06-30
Months Covered           : 66
Years Covered            : 6

🏆 SPENDING HIGHLIGHTS
Largest Expense Category : Rent — ₹305,176,701.60
Top Merchant             : House Rent — ₹305,176,701.60
Top City                 : Bhopal — ₹64,300,710.48
Average Expense          : ₹3,104.13
Median Expense           : ₹774.49

📈 MONTHLY PERFORMANCE
Highest Net Month        : December 2025 — ₹23,254,214.62
Lowest Net Month         : February 2024 — ₹17,019,365.36
Best Savings Rate        : 72.17%
Worst Savings Rate       : 63.84%

🗂️ DATABASE DIMENSIONS
Categories               : 19
Accounts                 : 8
Merchants         

In [13]:
CHARTS = {
    1:  ("Monthly Income vs Expense", chart_monthly_income_expense),
    2:  ("Monthly Net Cash Flow", chart_monthly_net),
    3:  ("Cumulative Cash Flow", chart_cumulative_cashflow),
    4:  ("Monthly Savings Rate", chart_savings_rate),
    5:  ("Expense by Category", chart_expense_category),
    6:  ("Expense Category Share", chart_expense_category_share),
    7:  ("Income by Category", chart_income_category),
    8:  ("Expense by Payment Method", chart_payment_method),
    9:  ("Expense by City", chart_city),
    10: ("Expense by Account", chart_account),
    11: ("Top 15 Merchants", chart_merchant),
    12: ("Expense by Weekday", chart_weekday),
    13: ("Monthly Transaction Count", chart_monthly_transaction_count),
    14: ("Monthly Expense Count", chart_monthly_expense_count),
    15: ("Average Expense per Transaction", chart_average_expense),
    16: ("Median Expense per Transaction", chart_median_expense),
    17: ("Expense Distribution", chart_expense_distribution),
    18: ("Income Distribution", chart_income_distribution),
    19: ("Quarterly Performance", chart_quarterly),
    20: ("Yearly Performance", chart_yearly),
    21: ("Category × Month Heatmap", chart_category_month_heatmap),
    22: ("Category Transaction Frequency", chart_category_frequency),
    23: ("Transaction Status", chart_status),
    24: ("Daily Expense Trend", chart_daily_expense),
    25: ("Daily Income Trend", chart_daily_income),
    26: ("20 Largest Expense Transactions", chart_top_transactions),
    27: ("Average Expense by Category", chart_category_average),
    28: ("Transactions by Account", chart_account_transaction_count),
    29: ("Transactions by City", chart_city_transaction_count),
    30: ("Income vs Expense by Category", chart_category_income_expense),
    31: ("30-Day Rolling Expense", chart_rolling_expense),
    32: ("Top Categories Over Time", chart_top_categories_month),
}


def print_menu():
    print("FINANCE DASHBOARD MENU\n")

    sections = [
        ("\nFINANCIAL OVERVIEW\n", range(1, 5)),
        ("\nSPENDING ANALYSIS\n", range(5, 13)),
        ("\nTRANSACTION ANALYSIS\n", range(13, 19)),
        ("\nTIME ANALYSIS\n", range(19, 26)),
        ("\nDETAILED ANALYSIS\n", range(26, 33)),
    ]

    for title, numbers in sections:
        print(f"{title}")
        for n in numbers:
            print(f"\t{n:>2}. {CHARTS[n][0]}")

    print("\n 0. Return / Exit")


def run_dashboard():
    while True:
        print_menu()
        choice = input("\nEnter dashboard option: ").strip()

        if choice == "0":
            print("\nDashboard session ended.")
            break

        try:
            choice = int(choice)
        except ValueError:
            print("Please enter a number.")
            continue

        if choice not in CHARTS:
            print("Invalid option. Choose 0–32.")
            continue

        title, function = CHARTS[choice]

        print(f"\nOpening: {title}")

        try:
            function()
        except Exception as e:
            print(f"Could not generate this chart: {e}")

        again = input("\nPress ENTER to return to the dashboard menu, or type Q to quit: ").strip().lower()
        if again == "q":
            print("Dashboard session ended.")
            break


run_dashboard()


FINANCE DASHBOARD MENU


FINANCIAL OVERVIEW

	 1. Monthly Income vs Expense
	 2. Monthly Net Cash Flow
	 3. Cumulative Cash Flow
	 4. Monthly Savings Rate

SPENDING ANALYSIS

	 5. Expense by Category
	 6. Expense Category Share
	 7. Income by Category
	 8. Expense by Payment Method
	 9. Expense by City
	10. Expense by Account
	11. Top 15 Merchants
	12. Expense by Weekday

TRANSACTION ANALYSIS

	13. Monthly Transaction Count
	14. Monthly Expense Count
	15. Average Expense per Transaction
	16. Median Expense per Transaction
	17. Expense Distribution
	18. Income Distribution

TIME ANALYSIS

	19. Quarterly Performance
	20. Yearly Performance
	21. Category × Month Heatmap
	22. Category Transaction Frequency
	23. Transaction Status
	24. Daily Expense Trend
	25. Daily Income Trend

DETAILED ANALYSIS

	26. 20 Largest Expense Transactions
	27. Average Expense by Category
	28. Transactions by Account
	29. Transactions by City
	30. Income vs Expense by Category
	31. 30-Day Rolling Expense
	32. To